# PrithviWORLD pipeline

**Can a frozen Earth-observation foundation model serve as the state layer of a world model?**

```
HLS  →  frozen Prithvi encoder  →  z_t (14×14×768)          the state
        z_t (+ Δt, day-of-year) →  dynamics                →  ẑ_{t+Δ}
        frozen Prithvi decoder                              figures only
```

This notebook supersedes `project_pipeline.ipynb` (ConvLSTM) and `project_s4d.ipynb` (S4D).
Both remain valid for their own tables; this one is the version this project reports, and it
differs in three ways that change the conclusions.

| Change | Why |
|---|---|
| **Multi-seed evaluation** | Re-training the *same* model moved residual MSE by 0.004, more than the gaps between models. Single-run rankings were not measurable. |
| **Climatology baseline** | Averaging several prior-year scenes near the target day-of-year, instead of copying one previous-year scene. This is the baseline the vegetation-forecasting literature compares against, and it is much harder to beat. |
| **Fixed Δ probe** | `delta_mode='constant'` used to read the batch mean, which at `batch_size=1` was each sample's own horizon. It now takes an explicit constant from the training windows. |

**Expected reading.** All learned models beat persistence comfortably. Whether any of them
beats the climatology is the actual question, and on the current split the honest answer is
reported in the results table.


## Section 1 — Setup

A **token** is one 16×16-pixel patch as a 768-D vector; a 224×224 window is **14×14 tokens**,
and that grid is the latent state `z`. Prithvi is never updated.

In [ ]:
from pathlib import Path
import sys, os, json, importlib, platform
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

for name in [
    "src.prithvi_hls", "src.data_utils", "src.paths", "src.prithvi_utils",
    "src.embedding_utils", "src.similarity_utils", "src.visualization_utils",
    "src.dynamics", "src.s4d", "src.rq1_analysis", "src.ndvi_utils",
]:
    importlib.reload(importlib.import_module(name))

from src.paths import (
    MODEL_DIR, RESULTS_DIR, PROJECT_DATA_DIR, PROJECT_EMBEDDINGS_DIR,
    ensure_project_dirs, load_config, location_hls_dir,
)
from src.prithvi_hls import index_hls_directory, lonlat_to_pixel, make_rgb_display, parse_hls_date
from src.prithvi_utils import decode_tokens_to_patch, denormalize_prithvi, load_single_frame_prithvi
from src.embedding_utils import list_embedding_keys, load_embedding, load_location_sequence
from src.visualization_utils import plot_change_triplet, plot_decoder_triplet
from src.dynamics import (
    ConvLSTMForecaster, ForecastTokenDataset, LinearDOYForecaster,
    TokenLSTMForecaster, baseline_rows, build_forecast_samples, change_maps_from_batch,
    evaluate_forecaster, make_weighted_loader, nuisance_floor, run_seeds,
    seed_everything, split_samples_by_year, summarize_seed_runs, train_forecaster,
)
from src.s4d import TokenS4DForecaster
from src.ndvi_utils import (
    compare_token_vs_ndvi_change, evaluate_ndvi_baselines, map_corr,
    ndvi_change_map, ndvi_grids_for_location, token_change_map,
)

SEED = 0
seed_everything(SEED)
pd.set_option("display.width", 160)

print("Python:", platform.python_version(), "| PyTorch:", torch.__version__,
      "| CUDA:", torch.cuda.is_available())
print("Repo:", REPO_ROOT.resolve())


### Flags

| Flag | Meaning |
|---|---|
| `SEEDS` | Every trained model runs once per seed. Fewer than 3 makes the table unreportable. |
| `MIN_DELTA_DAYS` | Forecast horizon floor. 21 days skips near-identity 5-day steps. |
| `LOAD_PRITHVI` | Only needed for the decoder figures. |
| `RUN_HEAVY` | `False` skips the multi-seed sweep and reloads the saved table. |


In [ ]:
CONFIG = load_config()
LOCATIONS = CONFIG["locations"]

RUN_LOCATION_IDS = ["location_01", "location_02"]   # Dataset B: California, Italy
HISTORY = 6
MIN_DELTA_DAYS = 21
MIN_CLEAR = 0.90                  # see Section 2; 0.50 lets undetected cloud dominate
CLIMATOLOGY_WINDOW_DAYS = 20      # +/- days around the target DOY
CLIMATOLOGY_PAST_ONLY = True      # never average in a year later than the target

SEEDS = (0, 1, 2, 3, 4)
EPOCHS = 40
BATCH_SIZE = 8
LR = 1e-3
PROJ_DIM = 128
HIDDEN = 96
S4D_STATE = 32

LOAD_PRITHVI = True
RUN_HEAVY = True
NDVI_FORCE = False

ensure_project_dirs(list(LOCATIONS.keys()))
PROJECT_DIR = RESULTS_DIR / "project"
FINAL_DIR = PROJECT_DIR / "final"
for d in [FINAL_DIR, FINAL_DIR / "figures", FINAL_DIR / "decoder",
          PROJECT_DIR / "checkpoints"]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def active_hls_dir(lid):
    return location_hls_dir(lid, data_root=PROJECT_DATA_DIR)

print("Device:", device)
for lid in RUN_LOCATION_IDS:
    loc = LOCATIONS[lid]
    n = len(list_embedding_keys(lid, embeddings_root=PROJECT_EMBEDDINGS_DIR))
    print(f"  {lid}: {loc['name']:<16s} ({loc['latitude']:7.3f}, {loc['longitude']:8.3f})  "
          f"{n:3d} embedded dates  [{loc['environment']}]")


## Section 2 — The nuisance floor, forecast windows, and the baseline ladder

### 3a. How much of the residual is noise?

Two cloud-free scenes 5 days apart contain almost no real land-surface change, so the residual
between them measures what no forecaster can predict: atmosphere, illumination, view geometry,
residual haze, co-registration. Every error below has to be read against that number.

It also audits the cloud screen. If the floor tracks `clear_fraction`, the "noise" is undetected
cloud rather than anything intrinsic, and the fix is a stricter threshold — not a better model.
This is what sets `MIN_CLEAR`, and it is the single most consequential choice in the pipeline:
a lenient threshold silently hands the benchmark to averaging baselines, because climatology
averages ~10 scenes and cancels the cloud noise while a learned model conditions on one noisy `z_t`.

### 3b. Windows and baselines

A window is `HISTORY` past states plus the first observation at least `MIN_DELTA_DAYS` later.
Split by the **target year**: 2022–2023 train, 2024 test.

Three parameter-free baselines, in increasing order of difficulty:

| Baseline | Prediction | What beating it proves |
|---|---|---|
| **persistence** | `ẑ = z_t` | the model predicts *something* changed |
| **seasonal copy** | nearest scene one year earlier | the model beats a single noisy seasonal draw |
| **climatology** | mean of prior-year scenes within ±20 days of the target DOY | the model knows something beyond the average season |

Climatology is the one that matters. It is strictly causal (`CLIMATOLOGY_PAST_ONLY`), so a
2024 target averages only 2022–2023 scenes.


In [ ]:
raw_sequences = {}
for lid in RUN_LOCATION_IDS:
    seq = load_location_sequence(lid, embeddings_root=PROJECT_EMBEDDINGS_DIR)
    if seq:
        raw_sequences[lid] = seq

def clear_fraction(item):
    cf = (item.get("metadata") or {}).get("clear_fraction")
    return float(cf) if cf is not None else 0.0

floor_tab = pd.DataFrame(nuisance_floor(raw_sequences, max_gap_days=7))
floor_tab["clear_min"] = [
    min(clear_fraction(it) for it in raw_sequences[r.location_id] if it["key"] in (r.key_a, r.key_b))
    for r in floor_tab.itertuples()
]
floor_tab["site"] = floor_tab["location_id"].map(lambda l: LOCATIONS[l]["name"])
floor_tab.to_csv(FINAL_DIR / "nuisance_floor.csv", index=False)

print(f"{len(floor_tab)} clear-scene pairs 5 days apart\n")
print(floor_tab.groupby("site")["residual_mse"].agg(["count", "mean", "median"]).round(4).to_string())
print(f"\ncorr(clear fraction, residual) = {floor_tab[['clear_min','residual_mse']].corr().iloc[0,1]:.3f}")
print("\nfloor as a function of the clear-sky threshold:")
for thr in [0.50, 0.70, 0.90, 0.95, 0.99]:
    sub = floor_tab[floor_tab["clear_min"] >= thr]
    kept = {lid: sum(clear_fraction(it) >= thr for it in s) for lid, s in raw_sequences.items()}
    print(f"  >={thr:.2f}: floor={sub['residual_mse'].mean():.4f}  "
          f"({len(sub):3d} pairs)  dates kept {kept}")

sequences = {lid: [it for it in s if clear_fraction(it) >= MIN_CLEAR]
             for lid, s in raw_sequences.items()}
sequences = {lid: s for lid, s in sequences.items() if len(s) > HISTORY + 1}
NUISANCE_FLOOR = float(floor_tab.loc[floor_tab["clear_min"] >= MIN_CLEAR, "residual_mse"].mean())
print(f"\nMIN_CLEAR={MIN_CLEAR:.2f} -> nuisance floor {NUISANCE_FLOOR:.4f}, "
      f"dates {({k: len(v) for k, v in sequences.items()})}")

# Tightening the screen is not free. EarthNet2021 (Sec 4.2) warns that thresholding on
# quality induces selection bias, because cloud is seasonal: they lost northern winters
# entirely. Report which months this costs rather than discovering it in the discussion.
keep_rows = []
for lid, seq in raw_sequences.items():
    for it in seq:
        keep_rows.append({"site": LOCATIONS[lid]["name"], "month": int(it["key"][5:7]),
                          "kept": clear_fraction(it) >= MIN_CLEAR})
keep = pd.DataFrame(keep_rows)
retention = (100 * keep.pivot_table(index="site", columns="month", values="kept",
                                    aggfunc="mean")).round(0)
retention.to_csv(FINAL_DIR / "min_clear_retention.csv")
print(f"\nPercent of dates surviving MIN_CLEAR={MIN_CLEAR:.2f}, by month:")
print(retention.to_string())
worst = retention.min(axis=1)
for site, v in worst.items():
    if v < 70:
        m = int(retention.loc[site].idxmin())
        print(f"  WARNING {site}: month {m} keeps only {v:.0f}% of its dates. "
              f"Seasonal coverage is biased; state this in the limitations.")

In [ ]:
samples = build_forecast_samples(
    sequences,
    history=HISTORY,
    min_delta_days=MIN_DELTA_DAYS,
    climatology_window_days=CLIMATOLOGY_WINDOW_DAYS,
    climatology_past_only=CLIMATOLOGY_PAST_ONLY,
)
train_s, val_s, test_s = split_samples_by_year(samples)

dts = [float(s.delta_days[-1]) for s in samples]
clim_n = [len(s.climatology_keys or []) for s in samples if s.has_climatology]
print(f"windows {len(samples)}  train={len(train_s)} val={len(val_s)} test={len(test_s)}")
print(f"horizon dt: mean={np.mean(dts):.1f}d  min={min(dts):.0f}  max={max(dts):.0f}")
print(f"seasonal copy available : {sum(s.has_seasonal for s in samples)}/{len(samples)}")
print(f"climatology available   : {sum(s.has_climatology for s in samples)}/{len(samples)}"
      f"  (mean {np.mean(clim_n):.1f} scenes averaged)")

TRAIN_MEAN_HORIZON = float(np.mean([s.delta_days[-1] for s in train_s]))
print(f"train mean horizon      : {TRAIN_MEAN_HORIZON:.1f} d   <- the delta-probe constant")

In [ ]:
# Dynamics use z, Δt, and day-of-year only (no exogenous forcing).
print(f"{len(samples)} forecast windows  |  train {len(train_s)}  val {len(val_s)}  test {len(test_s)}")


In [ ]:
base = pd.DataFrame(baseline_rows(test_s, device))
base.to_csv(FINAL_DIR / "baselines.csv", index=False)
print(f"Baselines on the {len(test_s)} test windows")
print(base[["model", "residual_mse", "change_map_corr", "aligned_cosine"]].round(4).to_string(index=False))

per_site_base = []
for lid in sorted({s.location_id for s in test_s}):
    sub = [s for s in test_s if s.location_id == lid]
    t = pd.DataFrame(baseline_rows(sub, device))
    t.insert(0, "site", LOCATIONS[lid]["name"])
    per_site_base.append(t)
    print(f"\n{LOCATIONS[lid]['name']} (n={len(sub)})")
    print(t[["model", "residual_mse", "change_map_corr"]].round(4).to_string(index=False))
pd.concat(per_site_base, ignore_index=True).to_csv(FINAL_DIR / "baselines_by_site.csv", index=False)

## Section 3 — Multi-seed model comparison

Four models. The contrasts are the experiment:

| Model | Memory | Spatial mixing | Continuous Δ |
|---|---|---|---|
| `linear` | – | – | – |
| `token-LSTM` | ✓ | – | – |
| `ConvLSTM` | ✓ | ✓ | – |
| `S4D` | ✓ | – | ✓ |

`linear` vs the rest isolates **memory and spatial mixing**. `token-LSTM` vs `S4D`
isolates **continuous time**.

Each is trained once per seed. Report mean ± std; a gap smaller than the standard
deviation is not a result.


In [ ]:
def build_factories():
    return {
        "linear": lambda: LinearDOYForecaster(proj_dim=PROJ_DIM, hidden=0),
        "token-LSTM": lambda: TokenLSTMForecaster(proj_dim=PROJ_DIM, hidden=HIDDEN),
        "ConvLSTM": lambda: ConvLSTMForecaster(proj_dim=PROJ_DIM, hidden=HIDDEN),
        "S4D": lambda: TokenS4DForecaster(
            proj_dim=PROJ_DIM, d_state=S4D_STATE, constant_delta=TRAIN_MEAN_HORIZON),
    }

seed_csv = FINAL_DIR / "seed_runs.csv"
if RUN_HEAVY:
    rows = run_seeds(
        build_factories(), train_s, val_s, test_s, device,
        seeds=SEEDS, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR,
    )
    seed_df = pd.DataFrame(rows)
    seed_df.to_csv(seed_csv, index=False)
else:
    seed_df = pd.read_csv(seed_csv)
    seed_df = seed_df[~seed_df["model"].astype(str).str.contains("weather", case=False)]
    print("loaded", seed_csv, "(weather models dropped if present)")

summary = pd.DataFrame(summarize_seed_runs(seed_df.to_dict("records")))
summary["spread"] = summary["residual_mse_max"] - summary["residual_mse_min"]
summary = summary.sort_values("residual_mse_mean")
summary.to_csv(FINAL_DIR / "seed_summary.csv", index=False)
summary[["model", "n_seeds", "residual_mse_mean", "residual_mse_std", "spread",
         "change_map_corr_mean", "change_map_corr_std"]].round(4)


In [ ]:
# The noise floor decides which comparisons are legible at all.
noise = float(summary["residual_mse_std"].max())
clim_mse = float(base.loc[base["model"] == "climatology", "residual_mse"].iloc[0])
pers_mse = float(base.loc[base["model"] == "persistence", "residual_mse"].iloc[0])
print(f"Largest across-seed std : {noise:.4f}   <- anything smaller is not a result")
print(f"Climatology             : {clim_mse:.4f}")
print(f"Persistence             : {pers_mse:.4f}")
print(f"Nuisance floor          : {NUISANCE_FLOOR:.4f}   <- no model can go below this")
best = float(summary["residual_mse_mean"].min())
signal = pers_mse - NUISANCE_FLOOR
print(f"\nOf the {pers_mse:.4f} residual energy at the horizon, {NUISANCE_FLOOR:.4f} "
      f"({100*NUISANCE_FLOOR/pers_mse:.0f}%) is nuisance and")
print(f"{signal:.4f} ({100*signal/pers_mse:.0f}%) is real change. The best model removes "
      f"{100*(pers_mse-best)/signal:.0f}% of the real part and sits")
print(f"at {best/NUISANCE_FLOOR:.2f}x the floor.\n")

for _, r in summary.iterrows():
    m, sd = r["residual_mse_mean"], r["residual_mse_std"]
    vs_clim = "beats climatology" if m + sd < clim_mse else (
        "ties climatology" if m - sd < clim_mse else "loses to climatology")
    print(f"  {r['model']:<16s} {m:.4f} ± {sd:.4f}   {vs_clim}")

print("\nPairwise contrasts (mean difference vs the noise floor):")
def contrast(a, b):
    ra = summary.loc[summary["model"] == a].iloc[0]
    rb = summary.loc[summary["model"] == b].iloc[0]
    d = ra["residual_mse_mean"] - rb["residual_mse_mean"]
    pooled = float(np.hypot(ra["residual_mse_std"], rb["residual_mse_std"]))
    verdict = "separable" if abs(d) > pooled else "inside noise"
    print(f"  {a:<16s} - {b:<16s} = {d:+.4f}  (pooled sd {pooled:.4f})  {verdict}")

for a, b in [("S4D", "token-LSTM"), ("ConvLSTM", "linear"), ("token-LSTM", "linear")]:
    contrast(a, b)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
order = summary.sort_values("residual_mse_mean")
ax.errorbar(order["residual_mse_mean"], range(len(order)),
            xerr=order["residual_mse_std"], fmt="o", capsize=4, color="#2b6cb0")
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order["model"])
for label, colour, style in [("persistence", "#a0aec0", ":"),
                             ("seasonal copy", "#f6ad55", "-."),
                             ("climatology", "#e53e3e", "--")]:
    v = float(base.loc[base["model"] == label, "residual_mse"].iloc[0])
    ax.axvline(v, color=colour, ls=style, lw=1.6, label=f"{label} ({v:.3f})")
ax.axvspan(0, NUISANCE_FLOOR, color="#cbd5e0", alpha=0.45, zorder=0)
ax.axvline(NUISANCE_FLOOR, color="#4a5568", lw=1.6,
           label=f"nuisance floor ({NUISANCE_FLOOR:.3f})")
ax.set_xlim(left=min(NUISANCE_FLOOR * 0.9, float(order["residual_mse_mean"].min()) * 0.95))
ax.set_xlabel("residual MSE on 2024 test windows (lower is better)")
ax.set_title(f"Latent forecasting, mean ± std over {len(SEEDS)} seeds")
ax.legend(fontsize=9, loc="lower right")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(FINAL_DIR / "figures" / "model_comparison.png", dpi=140, bbox_inches="tight")
plt.show()

## Section 4 — Does S4D use its clock?

The state-space model's justification over an LSTM is that Δ enters the discretization as
real elapsed days. The same trained weights are scored three ways:

| Δ fed at inference | Meaning |
|---|---|
| `true` | real inter-frame gaps and the real horizon |
| `unit` | every step is 1 day — an LSTM's discrete clock |
| `constant` | every step is the **training** mean horizon |

If `true` is no better than `constant`, the continuous-time machinery is not paying for
itself. The learned poles are plotted **weighted by |C|**: a mode with no output weight
contributes nothing however interesting its period looks.


In [ ]:
if RUN_HEAVY:
    seed_everything(SEED)
    probe_model = TokenS4DForecaster(
        proj_dim=PROJ_DIM, d_state=S4D_STATE, constant_delta=TRAIN_MEAN_HORIZON)
    train_forecaster(probe_model, make_weighted_loader(train_s, BATCH_SIZE, seed=SEED),
                     DataLoader(ForecastTokenDataset(val_s), batch_size=1) if val_s else None,
                     device, epochs=EPOCHS, lr=LR,
                     ckpt_path=PROJECT_DIR / "checkpoints" / "s4d_final.pt",
                     log=lambda _m: None)

    class _DeltaMode(torch.nn.Module):
        def __init__(self, inner, mode):
            super().__init__()
            self.inner, self.mode = inner, mode
        def forward(self, past, delta_days, doy_sincos, **kwargs):
            return self.inner(past, delta_days, doy_sincos,
                              delta_mode=self.mode, **kwargs)

    loader = DataLoader(ForecastTokenDataset(test_s), batch_size=1)
    probe_rows = []
    for mode in ("true", "unit", "constant"):
        r = evaluate_forecaster(_DeltaMode(probe_model, mode).to(device), loader, device)
        probe_rows.append({"delta_mode": mode, **r})
    probe = pd.DataFrame(probe_rows)
    probe.to_csv(FINAL_DIR / "delta_probe.csv", index=False)

    ts = pd.DataFrame(probe_model.ssm.timescales_days())
    ts.to_csv(FINAL_DIR / "s4d_timescales.csv", index=False)
else:
    probe = pd.read_csv(FINAL_DIR / "delta_probe.csv")
    ts = pd.read_csv(FINAL_DIR / "s4d_timescales.csv")

print(probe[["delta_mode", "residual_mse", "change_map_corr"]].round(4).to_string(index=False))
t = float(probe.loc[probe.delta_mode == "true", "residual_mse"].iloc[0])
u = float(probe.loc[probe.delta_mode == "unit", "residual_mse"].iloc[0])
c = float(probe.loc[probe.delta_mode == "constant", "residual_mse"].iloc[0])
print(f"\nconstant delta = {TRAIN_MEAN_HORIZON:.1f} days (from the training windows)")
if abs(t - u) < 1e-4:
    print("The SSM ignores delta entirely - it is a plain sequence mixer.")
elif t <= c:
    print("True elapsed time beats a constant: the continuous-time clock earns its place.")
else:
    print(f"True delta ({t:.4f}) is WORSE than a constant delta ({c:.4f}). The model is "
          "delta-sensitive but the real elapsed-time signal does not generalise; report this.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
sizes = 30 + 600 * ts["c_share"]
sc = axes[0].scatter(ts["tau_decay_days"], ts["period_days"], s=sizes,
                     c=ts["c_share"], cmap="viridis", alpha=0.85, edgecolor="k", linewidth=0.4)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("decay τ (days)"); axes[0].set_ylabel("oscillation period (days)")
axes[0].axhline(365, color="green", ls=":", lw=1, label="1 year")
axes[0].axhline(MIN_DELTA_DAYS, color="gray", ls="--", lw=1, label=f"{MIN_DELTA_DAYS}-day horizon")
axes[0].set_title("Learned poles, sized by output weight |C|")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
fig.colorbar(sc, ax=axes[0], label="share of |C|")

axes[1].bar(ts["mode"], ts["c_share"], color="#2b6cb0")
axes[1].set_xlabel("SSM mode"); axes[1].set_ylabel("share of total |C|")
axes[1].set_title("Which modes the model actually uses")
axes[1].grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FINAL_DIR / "figures" / "s4d_timescales.png", dpi=140, bbox_inches="tight")
plt.show()

top = ts.nlargest(5, "c_share")[["mode", "tau_decay_days", "period_days", "c_share"]]
print("Five most-used modes:")
print(top.round(2).to_string(index=False))

## Section 5 — NDVI control

`NDVI = (B8A − B04) / (B8A + B04)` on the same window, same Fmask, averaged to the same
14×14 cells. NDVI never enters any loss; this is purely a control.

Two questions. Does the token change map track the |ΔNDVI| map — if the correlation is very
high, Prithvi is an expensive NDVI. And do the baselines rank the same way in NDVI space —
if seasonality wins there too, the phenology result is a property of the landscape rather
than an artefact of the embedding.


In [ ]:
ndvi_dir = PROJECT_DIR / "ndvi"
ndvi_dir.mkdir(parents=True, exist_ok=True)
ndvi_by_key = {}
for lid in sorted(sequences):
    loc = LOCATIONS[lid]
    key_of_date_id = {}
    for item in sequences[lid]:
        did = (item.get("metadata") or {}).get("acquisition_date_id")
        if did:
            key_of_date_id[str(did)] = item["key"]
    grids = ndvi_grids_for_location(
        lon=loc["longitude"], lat=loc["latitude"], hls_dir=active_hls_dir(lid),
        date_ids=list(key_of_date_id), cache_path=ndvi_dir / f"{lid}_ndvi_cells.npz",
        force=NDVI_FORCE,
    )
    for did, grid in grids.items():
        if did in key_of_date_id:
            ndvi_by_key[(lid, key_of_date_id[did])] = grid
    print(f"{lid} {loc['name']}: {len(grids)}/{len(key_of_date_id)} NDVI grids")

corr_df = pd.DataFrame(compare_token_vs_ndvi_change(test_s, ndvi_by_key))
if len(corr_df):
    corr_df.to_csv(FINAL_DIR / "token_vs_ndvi_change.csv", index=False)
    print("\ncorr(token change map, abs-dNDVI map)")
    print(corr_df.groupby("location_id")["corr_token_vs_ndvi"]
          .agg(["mean", "median", "count"]).round(3).to_string())
    overall = float(corr_df["corr_token_vs_ndvi"].mean())
    print(f"  all sites mean = {overall:.3f}  ->",
          "token change is largely an NDVI signal" if overall > 0.7
          else "tokens carry change NDVI does not fully explain")

ndvi_base = pd.DataFrame(evaluate_ndvi_baselines(test_s, ndvi_by_key))
if len(ndvi_base):
    ndvi_base.to_csv(FINAL_DIR / "ndvi_baselines.csv", index=False)
    print("\nNDVI-space baselines (MSE in NDVI units)")
    print(ndvi_base.round(4).to_string(index=False))

## Section 6 — Change maps

Where did the model say the landscape would move, against where it actually moved, against
what the climatology predicted. The window shown is the one with the highest real residual
energy — the largest genuine change in the test year.


In [ ]:
def res_energy(s):
    return float(np.mean((s.target - s.past[-1]) ** 2))

viz_by_site = {}
for s in test_s:
    cur = viz_by_site.get(s.location_id)
    if cur is None or res_energy(s) > res_energy(cur):
        viz_by_site[s.location_id] = s

if RUN_HEAVY:
    seed_everything(SEED)
    best_name = summary.iloc[0]["model"]
    best_model = build_factories()[best_name]()
    train_forecaster(best_model, make_weighted_loader(train_s, BATCH_SIZE, seed=SEED),
                     DataLoader(ForecastTokenDataset(val_s), batch_size=1) if val_s else None,
                     device, epochs=EPOCHS, lr=LR, log=lambda _m: None)
    print("figure model:", best_name)

    for lid, s in viz_by_site.items():
        past = torch.from_numpy(s.past).unsqueeze(0).to(device)
        dt = torch.from_numpy(s.delta_days).unsqueeze(0).to(device)
        doy = torch.from_numpy(s.doy_sincos).unsqueeze(0).to(device)
        tgt = torch.from_numpy(s.target).unsqueeze(0).to(device)
        clim = torch.from_numpy(np.asarray(s.climatology, dtype=np.float32)).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = best_model(past, dt, doy)
        m_model = change_maps_from_batch(pred, tgt, past[:, -1])
        m_clim = change_maps_from_batch(clim, tgt, past[:, -1])

        fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
        panels = [(m_clim["predicted_change"], "Climatology Δ"),
                  (m_model["predicted_change"], f"{best_name} Δ"),
                  (m_model["real_change"], "Real Δ")]
        vmax = max(float(p.max()) for p, _ in panels)
        for ax, (im, lab) in zip(axes, panels):
            h = ax.imshow(im, cmap="magma", vmin=0, vmax=vmax)
            ax.set_title(lab, fontsize=10); ax.axis("off")
            fig.colorbar(h, ax=ax, fraction=0.046)
        fig.suptitle(f"{LOCATIONS[lid]['name']}  {s.past_keys[-1]} → {s.target_key}"
                     f"   (Δt={int(s.delta_days[-1])}d)")
        fig.savefig(FINAL_DIR / "figures" / f"change_{lid}.png", dpi=140, bbox_inches="tight")
        plt.show()


## Section 7 — Decoder viewer (figures only)

The frozen MAE decoder maps a full token grid back toward a 6-band patch. It was trained to
fill holes in the *current* image, not to paint the future, so this is a picture of `ẑ` and
nothing more. Never train through it.

If the forecast panel looks identical to the persistence panel, no change was forecast.


In [ ]:
if not LOAD_PRITHVI:
    print("Skip decoder — set LOAD_PRITHVI=True.")
else:
    bundle = load_single_frame_prithvi(repo_id=CONFIG["model"].get("repo_id"), device=device)
    model = bundle["model"]
    print("Prithvi on", bundle["device"], "params", bundle["num_parameters"])

    s = max(test_s, key=res_energy)
    loc = LOCATIONS[s.location_id]
    last_item = load_embedding(s.location_id, s.past_keys[-1], embeddings_root=PROJECT_EMBEDDINGS_DIR)
    tgt_item = load_embedding(s.location_id, s.target_key, embeddings_root=PROJECT_EMBEDDINGS_DIR)

    def ydoy(item):
        did = item["metadata"].get("acquisition_date_id")
        return parse_hls_date(str(did)) if did else (2024, 180)

    y_t, d_t = ydoy(last_item)
    y_n, d_n = ydoy(tgt_item)
    lat, lon = loc["latitude"], loc["longitude"]

    past = torch.from_numpy(s.past).unsqueeze(0).to(device)
    dt = torch.from_numpy(s.delta_days).unsqueeze(0).to(device)
    doy = torch.from_numpy(s.doy_sincos).unsqueeze(0).to(device)
    with torch.no_grad():
        pred_tok = best_model(past, dt, doy)[0].cpu().numpy()

    cls_last = last_item["cls"]
    rgb = lambda p: make_rgb_display(denormalize_prithvi(p))
    p_real = rgb(decode_tokens_to_patch(model, device, tgt_item["tokens"], tgt_item["cls"], y_n, d_n, lat, lon))
    p_pred = rgb(decode_tokens_to_patch(model, device, pred_tok, cls_last, y_n, d_n, lat, lon))
    p_last = rgb(decode_tokens_to_patch(model, device, last_item["tokens"], cls_last, y_t, d_t, lat, lon))

    out_dir = FINAL_DIR / "decoder"
    plot_decoder_triplet(p_real, p_pred, p_last,
                         f"{loc['name']}  {s.past_keys[-1]} → {s.target_key}",
                         out_dir / f"{s.location_id}_triplet.png")
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, im, lab in zip(axes, [p_last, p_pred, p_real],
                           [f"Decode z_t\n{s.past_keys[-1]}",
                            f"Decode forecast ẑ\n{s.target_key}",
                            f"Decode real z\n{s.target_key}"]):
        ax.imshow(np.clip(im, 0, 1)); ax.set_title(lab, fontsize=10); ax.axis("off")
    fig.suptitle(f"{loc['name']} — frozen MAE decoder")
    fig.savefig(out_dir / f"{s.location_id}_triplet_inline.png", dpi=140, bbox_inches="tight")
    plt.show()


## Section 8 — The results table

One table for this project. Parameter-free baselines have no seed spread.


In [ ]:
rows = []
for _, r in base.iterrows():
    rows.append({"model": r["model"], "kind": "baseline", "n_seeds": 0,
                 "residual_mse": f"{r['residual_mse']:.4f}",
                 "change_map_corr": f"{r['change_map_corr']:.3f}",
                 "residual_mse_value": r["residual_mse"]})
for _, r in summary.iterrows():
    rows.append({"model": r["model"], "kind": "learned", "n_seeds": int(r["n_seeds"]),
                 "residual_mse": f"{r['residual_mse_mean']:.4f} ± {r['residual_mse_std']:.4f}",
                 "change_map_corr": f"{r['change_map_corr_mean']:.3f} ± {r['change_map_corr_std']:.3f}",
                 "residual_mse_value": r["residual_mse_mean"]})

final = pd.DataFrame(rows).sort_values("residual_mse_value").drop(columns="residual_mse_value")
final.to_csv(FINAL_DIR / "results_table.csv", index=False)
print(f"Test year 2024, {len(test_s)} windows, {len(SEEDS)} seeds per learned model\n")
print(final.to_string(index=False))
print(f"\nNoise floor (largest across-seed std): {noise:.4f}")
print("Gaps smaller than that are not results.")
print("\nSaved to", FINAL_DIR)

## What this notebook produces

| Path | Content |
|---|---|
| `results/project/final/results_table.csv` | The headline table |
| `results/project/final/seed_runs.csv` | Every (model, seed) run |
| `results/project/final/seed_summary.csv` | Mean ± std and spread |
| `results/project/final/baselines.csv` | Persistence / seasonal / climatology |
| `results/project/final/delta_probe.csv` | true / unit / constant Δ |
| `results/project/final/s4d_timescales.csv` | Poles with `c_magnitude` and `c_share` |
| `results/project/final/figures/` | Model comparison, timescales, change maps |

**Run order.** Top to bottom. Set `RUN_HEAVY = False` to reload saved tables and only redraw
figures.
